# Feature Engineering
Pada tahap ini dilakukan pembuatan fitur baru dari data pertandingan sepak bola internasional. Fitur yang dibuat akan digunakan sebagai input model machine learning untuk memprediksi hasil pertandingan

## Import Library

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

## Load Dataset Hasil Preprocessing

In [9]:
path = Path("../data/processed/match_processed.csv")
df = pd.read_csv(path)

print("Dataset berhasil dibaca dari:", path)
print("Jumlah baris:", df.shape[0])
print("Jumlah kolom:", df.shape[1])

df.head()

Dataset berhasil dibaca dari: ..\data\processed\match_processed.csv
Jumlah baris: 25410
Jumlah kolom: 11


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,result
0,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False,2000,home_win
1,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False,2000,home_win
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False,2000,draw
3,2000-01-09,Burkina Faso,Gabon,1.0,1.0,Friendly,Ouagadougou,Burkina Faso,False,2000,draw
4,2000-01-09,Guatemala,Armenia,1.0,1.0,Friendly,Los Angeles,United States,True,2000,draw


## Menyiapkan Data Pertandingan
Kolom tanggal diubah ke format datetime dan data diurutkan berdasarkan tanggal pertandingan agar fitur performa sebelumnya dapat dihitung dengan benar.

In [11]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df["Match_id"] = df.index
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,result,Match_id
0,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False,2000,home_win,0
1,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False,2000,home_win,1
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False,2000,draw,2
3,2000-01-09,Burkina Faso,Gabon,1.0,1.0,Friendly,Ouagadougou,Burkina Faso,False,2000,draw,3
4,2000-01-09,Guatemala,Armenia,1.0,1.0,Friendly,Los Angeles,United States,True,2000,draw,4


## Membuat Data Performa Setiap Tim

Dataset pertandingan akan diubah menjadi format per tim. Setiap pertandingan menghasilkan dua baris data, yaitu satu untuk home team dan satu untuk away team.

In [20]:
home_data = df[[
    "Match_id", "date", "home_team", "away_team", 
    "home_score", "away_score", "tournament", "neutral"
]].copy()

home_data.columns = [
    "Match_id", "date", "team", "opponent", "goals_for", "goals_against", "tournament", "neutral"
]

home_data["is_home"] = 1

away_data = df[[
    "Match_id", "date", "away_team", "home_team",
    "away_score", "home_score", "tournament", "neutral"
]].copy()

away_data.columns = [
    "Match_id", "date", "team", "opponent",
    "goals_for", "goals_against", "tournament", "neutral"
]

away_data["is_home"] = 0

team_matches = pd.concat([home_data, away_data], ignore_index=True)
team_matches = team_matches.sort_values(["team", "date", "Match_id"]).reset_index(drop=True)
team_matches.head()

,Match_id,date,team,opponent,goals_for,goals_against,tournament,neutral,is_home
0,12196,2012-09-25,Abkhazia,Artsakh,1.0,1.0,Friendly,False,1
1,12342,2012-10-21,Abkhazia,Artsakh,0.0,3.0,Friendly,False,0
2,13255,2013-09-23,Abkhazia,South Ossetia,3.0,0.0,Friendly,False,1
3,13719,2014-06-01,Abkhazia,Occitania,1.0,1.0,CONIFA World Football Cup,True,1
4,13736,2014-06-02,Abkhazia,Sápmi,2.0,1.0,CONIFA World Football Cup,False,0


## Membuah Kolom Hasil Tim

Pada bagian ini dibuat kolom hasil pertandingan dari sudut pandang masing-masing tim, seperti menang, seri, kalah, serta poin yang didapat.

In [23]:
def get_team_result(row):
    if row["goals_for"] > row["goals_against"]:
        return "win"
    elif row["goals_for"] < row["goals_against"]:
        return "lose"
    else:
        return "draw"

team_matches["team_result"] = team_matches.apply(get_team_result, axis=1)

team_matches["points"] = team_matches["team_result"].map({
    "win":3,
    "draw":1,
    "lose":0
})

team_matches["win"] = (team_matches["team_result"] == "win").astype(int)
team_matches["draw"] = (team_matches["team_result"] == "draw").astype(int)
team_matches["lose"] = (team_matches["team_result"] == "lose").astype(int)

team_matches.head()


,Match_id,date,team,opponent,goals_for,goals_against,tournament,neutral,is_home,team_result,points,win,draw,lose
0,12196,2012-09-25,Abkhazia,Artsakh,1.0,1.0,Friendly,False,1,draw,1,0,1,0
1,12342,2012-10-21,Abkhazia,Artsakh,0.0,3.0,Friendly,False,0,lose,0,0,0,1
2,13255,2013-09-23,Abkhazia,South Ossetia,3.0,0.0,Friendly,False,1,win,3,1,0,0
3,13719,2014-06-01,Abkhazia,Occitania,1.0,1.0,CONIFA World Football Cup,True,1,draw,1,0,1,0
4,13736,2014-06-02,Abkhazia,Sápmi,2.0,1.0,CONIFA World Football Cup,False,0,win,3,1,0,0


## Membuat fitur Performa 5 Pertandingan Terakhir
Fitur ini dibuat berdasarkan rata-rata performa masing-masing tim pada 5 pertandingan sebelumnya.

In [24]:
rolling_window = 5

team_matches["matches_before"] = team_matches.groupby("team").cumcount()

team_matches["goals_for_last5"] = (
    team_matches
    .groupby("team")["goals_for"]
    .transform(lambda x: x.shift(1).rolling(rolling_window, min_periods=1).mean())
)

team_matches["goals_against_last5"] = (
    team_matches
    .groupby("team")["goals_against"]
    .transform(lambda x: x.shift(1).rolling(rolling_window, min_periods=1).mean())
)

team_matches["points_last5"] = (
    team_matches
    .groupby("team")["points"]
    .transform(lambda x: x.shift(1).rolling(rolling_window, min_periods=1).mean())
)

team_matches["win_rate_last5"] = (
    team_matches
    .groupby("team")["win"]
    .transform(lambda x: x.shift(1).rolling(rolling_window, min_periods=1).mean())
)

team_matches.head(10)

,Match_id,date,team,opponent,goals_for,goals_against,tournament,neutral,is_home,team_result,points,win,draw,lose,matches_before,goals_for_last5,goals_against_last5,points_last5,win_rate_last5
0,12196,2012-09-25,Abkhazia,Artsakh,1.0,1.0,Friendly,False,1,draw,1,0,1,0,0,NaN,NaN,NaN,NaN
1,12342,2012-10-21,Abkhazia,Artsakh,0.0,3.0,Friendly,False,0,lose,0,0,0,1,1,1.000000,1.000000,1.000000,0.000000
2,13255,2013-09-23,Abkhazia,South Ossetia,3.0,0.0,Friendly,False,1,win,3,1,0,0,2,0.500000,2.000000,0.500000,0.000000
3,13719,2014-06-01,Abkhazia,Occitania,1.0,1.0,CONIFA World Football Cup,True,1,draw,1,0,1,0,3,1.333333,1.333333,1.333333,0.333333
4,13736,2014-06-02,Abkhazia,Sápmi,2.0,1.0,CONIFA World Football Cup,False,0,win,3,1,0,0,4,1.250000,1.250000,1.250000,0.250000
5,13751,2014-06-04,Abkhazia,South Ossetia,0.0,0.0,CONIFA World Football Cup,True,1,draw,1,0,1,0,5,1.400000,1.200000,1.600000,0.400000
6,13765,2014-06-05,Abkhazia,Padania,3.0,3.0,CONIFA World Football Cup,True,0,draw,1,0,1,0,6,1.200000,1.000000,1.600000,0.400000
7,13782,2014-06-07,Abkhazia,Occitania,0.0,1.0,CONIFA World Football Cup,True,1,lose,0,0,0,1,7,1.800000,1.000000,1.800000,0.400000
8,14580,2015-05-05,Abkhazia,Luhansk PR,1.0,0.0,Friendly,False,1,win,3,1,0,0,8,1.200000,1.200000,1.200000,0.200000
9,14588,2015-05-14,Abkhazia,Donetsk PR,1.0,0.0,Friendly,False,1,win,3,1,0,0,9,1.200000,1.000000,1.600000,0.400000


## Menggabungkan Fitur ke Dataset Utama
Fitur performa home team dan away team digabungkan kembali ke dataset pertandingan utama.

In [25]:
feature_columns = [
    "Match_id",
    "goals_for_last5",
    "goals_against_last5",
    "points_last5",
    "win_rate_last5",
    "matches_before"
]

home_features = team_matches[team_matches["is_home"] == 1][feature_columns].copy()
away_features = team_matches[team_matches["is_home"] == 0][feature_columns].copy()

home_features = home_features.rename(columns={
    "goals_for_last5": "home_goals_for_last5",
    "goals_against_last5": "home_goals_against_last5",
    "points_last5": "home_points_last5",
    "win_rate_last5": "home_win_rate_last5",
    "matches_before": "home_matches_before"
})

away_features = away_features.rename(columns={
    "goals_for_last5": "away_goals_for_last5",
    "goals_against_last5": "away_goals_against_last5",
    "points_last5": "away_points_last5",
    "win_rate_last5": "away_win_rate_last5",
    "matches_before": "away_matches_before"
})

df_features = df.merge(home_features, on="Match_id", how="left")
df_features = df_features.merge(away_features, on="Match_id", how="left")

df_features.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,...,home_goals_for_last5,home_goals_against_last5,home_points_last5,home_win_rate_last5,home_matches_before,away_goals_for_last5,away_goals_against_last5,away_points_last5,away_win_rate_last5,away_matches_before
0,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False,2000,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
1,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False,2000,...,NaN,NaN,NaN,NaN,0,1.0,2.0,0.0,0.0,1
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False,2000,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
3,2000-01-09,Burkina Faso,Gabon,1.0,1.0,Friendly,Ouagadougou,Burkina Faso,False,2000,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0
4,2000-01-09,Guatemala,Armenia,1.0,1.0,Friendly,Los Angeles,United States,True,2000,...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0


## Mengisi Missing Value pada Fitur Awal
Beberapa tim mungkin belum memiliki 5 pertandingan sebelumnya, sehingga nilai fiturnya kosong. Nilai kosong tersebut akan diisi dengan rata-rata umum.

In [26]:
feature_cols = [
    "home_goals_for_last5",
    "home_goals_against_last5",
    "home_points_last5",
    "home_win_rate_last5",
    "away_goals_for_last5",
    "away_goals_against_last5",
    "away_points_last5",
    "away_win_rate_last5"
]

for col in feature_cols:
    df_features[col] = df_features[col].fillna(df_features[col].mean())

df_features["home_matches_before"] = df_features["home_matches_before"].fillna(0)
df_features["away_matches_before"] = df_features["away_matches_before"].fillna(0)

df_features[feature_cols].isnull().sum()

home_goals_for_last5        0
home_goals_against_last5    0
home_points_last5           0
home_win_rate_last5         0
away_goals_for_last5        0
away_goals_against_last5    0
away_points_last5           0
away_win_rate_last5         0
dtype: int64

## Membuat Fitur Selisi Performa
Fitur selisih dibuat untuk membandingkan kekuatan home team dan away team berdasarkan performa sebelumnya.

In [27]:
df_features["goals_for_diff_last5"] = (
    df_features["home_goals_for_last5"] - df_features["away_goals_for_last5"]
)

df_features["goals_against_diff_last5"] = (
    df_features["home_goals_against_last5"] - df_features["away_goals_against_last5"]
)

df_features["points_diff_last5"] = (
    df_features["home_points_last5"] - df_features["away_points_last5"]
)

df_features["win_rate_diff_last5"] = (
    df_features["home_win_rate_last5"] - df_features["away_win_rate_last5"]
)

df_features["experience_diff"] = (
    df_features["home_matches_before"] - df_features["away_matches_before"]
)

df_features.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,year,...,away_goals_for_last5,away_goals_against_last5,away_points_last5,away_win_rate_last5,away_matches_before,goals_for_diff_last5,goals_against_diff_last5,points_diff_last5,win_rate_diff_last5,experience_diff
0,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False,2000,...,1.358042,1.401015,1.364099,0.376631,0,0.042391,-0.052502,0.040798,0.013981,0
1,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False,2000,...,1.000000,2.000000,0.000000,0.000000,1,0.400433,-0.651487,1.404896,0.390611,-1
2,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False,2000,...,1.358042,1.401015,1.364099,0.376631,0,0.042391,-0.052502,0.040798,0.013981,0
3,2000-01-09,Burkina Faso,Gabon,1.0,1.0,Friendly,Ouagadougou,Burkina Faso,False,2000,...,1.358042,1.401015,1.364099,0.376631,0,0.042391,-0.052502,0.040798,0.013981,0
4,2000-01-09,Guatemala,Armenia,1.0,1.0,Friendly,Los Angeles,United States,True,2000,...,1.358042,1.401015,1.364099,0.376631,0,0.042391,-0.052502,0.040798,0.013981,0


## Memilih Kolom untuk Model
Pada bagian ini dipilih kolom-kolom yang akan digunakan sebagai fitur model machine learning.

In [28]:
model_columns = [
    "date",
    "home_team",
    "away_team",
    "tournament",
    "neutral",
    "home_goals_for_last5",
    "home_goals_against_last5",
    "home_points_last5",
    "home_win_rate_last5",
    "away_goals_for_last5",
    "away_goals_against_last5",
    "away_points_last5",
    "away_win_rate_last5",
    "goals_for_diff_last5",
    "goals_against_diff_last5",
    "points_diff_last5",
    "win_rate_diff_last5",
    "experience_diff",
    "result"
]

df_model = df_features[model_columns].copy()

df_model.head()

,date,home_team,away_team,tournament,neutral,home_goals_for_last5,home_goals_against_last5,home_points_last5,home_win_rate_last5,away_goals_for_last5,away_goals_against_last5,away_points_last5,away_win_rate_last5,goals_for_diff_last5,goals_against_diff_last5,points_diff_last5,win_rate_diff_last5,experience_diff,result
0,2000-01-04,Egypt,Togo,Friendly,False,1.400433,1.348513,1.404896,0.390611,1.358042,1.401015,1.364099,0.376631,0.042391,-0.052502,0.040798,0.013981,0,home_win
1,2000-01-07,Tunisia,Togo,Friendly,False,1.400433,1.348513,1.404896,0.390611,1.000000,2.000000,0.000000,0.000000,0.400433,-0.651487,1.404896,0.390611,-1,home_win
2,2000-01-08,Trinidad and Tobago,Canada,Friendly,False,1.400433,1.348513,1.404896,0.390611,1.358042,1.401015,1.364099,0.376631,0.042391,-0.052502,0.040798,0.013981,0,draw
3,2000-01-09,Burkina Faso,Gabon,Friendly,False,1.400433,1.348513,1.404896,0.390611,1.358042,1.401015,1.364099,0.376631,0.042391,-0.052502,0.040798,0.013981,0,draw
4,2000-01-09,Guatemala,Armenia,Friendly,True,1.400433,1.348513,1.404896,0.390611,1.358042,1.401015,1.364099,0.376631,0.042391,-0.052502,0.040798,0.013981,0,draw


## Menyimpan Dataset Feature Engineering
Dataset yang sudah memiliki fitur baru akan disimpan untuk digunakan pada tahap training model.

In [29]:
output_path = Path("../data/processed/features_matches.csv")

output_path.parent.mkdir(parents=True, exist_ok=True)

df_model.to_csv(output_path, index=False)

print("Dataset feature engineering berhasil disimpan ke:", output_path)
print("Jumlah baris:", df_model.shape[0])
print("Jumlah kolom:", df_model.shape[1])

df_model.head()

Dataset feature engineering berhasil disimpan ke: ..\data\processed\features_matches.csv
Jumlah baris: 25410
Jumlah kolom: 19


,date,home_team,away_team,tournament,neutral,home_goals_for_last5,home_goals_against_last5,home_points_last5,home_win_rate_last5,away_goals_for_last5,away_goals_against_last5,away_points_last5,away_win_rate_last5,goals_for_diff_last5,goals_against_diff_last5,points_diff_last5,win_rate_diff_last5,experience_diff,result
0,2000-01-04,Egypt,Togo,Friendly,False,1.400433,1.348513,1.404896,0.390611,1.358042,1.401015,1.364099,0.376631,0.042391,-0.052502,0.040798,0.013981,0,home_win
1,2000-01-07,Tunisia,Togo,Friendly,False,1.400433,1.348513,1.404896,0.390611,1.000000,2.000000,0.000000,0.000000,0.400433,-0.651487,1.404896,0.390611,-1,home_win
2,2000-01-08,Trinidad and Tobago,Canada,Friendly,False,1.400433,1.348513,1.404896,0.390611,1.358042,1.401015,1.364099,0.376631,0.042391,-0.052502,0.040798,0.013981,0,draw
3,2000-01-09,Burkina Faso,Gabon,Friendly,False,1.400433,1.348513,1.404896,0.390611,1.358042,1.401015,1.364099,0.376631,0.042391,-0.052502,0.040798,0.013981,0,draw
4,2000-01-09,Guatemala,Armenia,Friendly,True,1.400433,1.348513,1.404896,0.390611,1.358042,1.401015,1.364099,0.376631,0.042391,-0.052502,0.040798,0.013981,0,draw
